In [6]:
import sys
from pathlib import Path
src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path))

In [13]:
from api.events.models import EventModel
from api.db.session import engine
from sqlmodel import Session, select
from timescaledb.hyperfunctions import time_bucket
from pprint import pprint

In [10]:
with Session(engine) as session:
    query = select(EventModel).order_by(EventModel.updated_at.desc()).limit(10)
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    print(compiled_query)
    print("")
    print(str(query))
    # results = session.exec(query).all()
    # print(results)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT 10

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT :param_1


In [18]:
from sqlalchemy import func
from datetime import datetime, timedelta, timezone

with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)
    pages = ['/about', '/contact', '/pages', '/pricing']
    start = datetime.now(timezone.utc) - timedelta(days=1)
    finish = datetime.now(timezone.utc)
    pprint(bucket)
    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(
            EventModel.time >= start,
            EventModel.time <= finish,
            EventModel.page.in_(pages))
        .group_by(bucket, EventModel.page)
        .order_by(bucket.desc(), EventModel.page )
    )
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)
 

<sqlalchemy.sql.functions.Function at 0x1d6d8fef290; time_bucket>
[(datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/about', 257),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/contact', 258),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/pages', 254),
 (datetime.datetime(2026, 1, 16, 0, 0, tzinfo=datetime.timezone.utc), '/pricing', 231)]
